In [1]:
!nvidia-smi

Thu Apr 16 11:23:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install transformers accelerate sentencepiece

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)
model = model.to("cuda")
model.eval()

print("Loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded successfully


In [5]:
prompt = "Explain the role of attention in transformer models."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        use_cache=True
    )

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Explain the role of attention in transformer models. In a transformer model, attention is used to focus on relevant parts of the input data and improve the performance of the model. Attention mechanisms are typically applied at the


In [ ]:
%%writefile benchmark_sweep_skeleton.py
# benchmark_day4_skeleton.py
# Day 4 - Single-request benchmark sweep
# Goal:
#   1. sweep prompt length
#   2. sweep output length
#   3. measure TTFT and TPOT
#   4. plot TTFT vs prompt length
#   5. plot TPOT vs output length
#
# NOTE:
#   Core logic is intentionally left as TODO.

import time
import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any

import torch
import matplotlib.pyplot as plt

# TODO:
# import HF classes you need
# from transformers import ...


# =========================
# Config
# =========================

@dataclass
class SweepConfig:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: str = "float16"

    # Fixed single-request setting
    concurrency: int = 1

    # Sweeps
    prompt_lengths: List[int] = None
    output_lengths: List[int] = None

    # Repetition
    num_runs: int = 3
    num_warmup: int = 1

    # Generation behavior
    do_sample: bool = False
    temperature: float = 1.0
    top_p: float = 1.0
    use_kv_cache: bool = True

    # Misc
    results_dir: str = "./results_day4"

    def __post_init__(self):
        if self.prompt_lengths is None:
            self.prompt_lengths = [32, 128, 512, 1024]
        if self.output_lengths is None:
            self.output_lengths = [32, 64, 128, 256]


# =========================
# Basic utils
# =========================

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)


def resolve_torch_dtype(dtype_str: str):
    mapping = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    if dtype_str not in mapping:
        raise ValueError(f"Unsupported dtype: {dtype_str}")
    return mapping[dtype_str]


# =========================
# Prompt helpers
# =========================

def build_base_prompt() -> str:
    """
    Return a reusable base prompt text.
    """
    # TODO:
    # return a base instruction prompt
    raise NotImplementedError


def make_prompt_near_target_tokens(
    tokenizer,
    base_text: str,
    target_prompt_len: int,
    max_iters: int = 100,
) -> str:
    """
    Construct a prompt whose tokenized length is close to target_prompt_len.
    """
    # TODO:
    # similar to your Day 3 version
    raise NotImplementedError


def count_input_tokens(tokenizer, prompt: str) -> int:
    """
    Return number of input tokens.
    """
    # TODO
    raise NotImplementedError


# =========================
# Model loading
# =========================

def load_tokenizer_and_model(cfg: SweepConfig):
    """
    Load tokenizer and model.
    """
    torch_dtype = resolve_torch_dtype(cfg.dtype)

    # TODO:
    # 1. tokenizer = ...
    # 2. model = ...
    # 3. set pad token if needed
    # 4. move model to device
    # 5. model.eval()
    raise NotImplementedError


def prepare_inputs(tokenizer, prompt: str, device: str):
    """
    Tokenize prompt and move to device.
    """
    # TODO
    raise NotImplementedError


# =========================
# Timing / measurement
# =========================

def run_prefill_and_decode_measurement(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int,
    cfg: SweepConfig,
) -> Dict[str, Any]:
    """
    Measure one single-request run.

    IMPORTANT:
    For Day 4, you want at least:
      - input_tokens
      - generated_tokens
      - total_latency_sec
      - ttft_sec
      - tpot_sec

    Definitions:
      - TTFT: time to first generated token
      - TPOT: average time per output token AFTER first token
               common approximation:
               (total_latency - ttft) / max(generated_tokens - 1, 1)

    You can implement this in different ways:
      A) use streamer/callback to detect first token arrival
      B) do manual token-by-token loop
      C) first implement a placeholder approximation, then improve it later

    Leave core logic for yourself.
    """
    inputs = prepare_inputs(tokenizer, prompt, cfg.device)

    # Optional but recommended
    if cfg.device == "cuda":
        torch.cuda.synchronize()

    # TODO:
    # Implement measurement logic.
    #
    # Suggested outputs:
    # {
    #   "input_tokens": ...,
    #   "generated_tokens": ...,
    #   "total_latency_sec": ...,
    #   "ttft_sec": ...,
    #   "tpot_sec": ...,
    #   "generated_text": ...
    # }
    #
    # Notes:
    # - If you use a manual decode loop, you can directly measure first-step latency
    # - If you use generate()+streaming, capture first token timestamp
    # - If generated_tokens <= 1, define TPOT carefully
    raise NotImplementedError


# =========================
# Warmup
# =========================

def run_warmup(
    model,
    tokenizer,
    prompt: str,
    warmup_output_len: int,
    cfg: SweepConfig,
):
    for i in range(cfg.num_warmup):
        print(f"[Warmup {i+1}/{cfg.num_warmup}]")
        _ = run_prefill_and_decode_measurement(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=warmup_output_len,
            cfg=cfg,
        )


# =========================
# Sweep 1: TTFT vs prompt length
# =========================

def run_prompt_length_sweep(
    model,
    tokenizer,
    cfg: SweepConfig,
    fixed_output_len: int = 64,
) -> List[Dict[str, Any]]:
    """
    Sweep prompt lengths while fixing output length.
    Goal: TTFT vs prompt length
    """
    records = []

    base_prompt = build_base_prompt()

    for prompt_len in cfg.prompt_lengths:
        prompt = make_prompt_near_target_tokens(
            tokenizer=tokenizer,
            base_text=base_prompt,
            target_prompt_len=prompt_len,
        )

        actual_input_tokens = count_input_tokens(tokenizer, prompt)

        print(f"\n=== Prompt sweep: target={prompt_len}, actual={actual_input_tokens} ===")

        # optional: warmup per setting
        # TODO:
        # decide whether to warm up once globally or once per setting

        for run_idx in range(cfg.num_runs):
            print(f"[Prompt sweep run {run_idx+1}/{cfg.num_runs}]")

            record = run_prefill_and_decode_measurement(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                max_new_tokens=fixed_output_len,
                cfg=cfg,
            )

            record["sweep_type"] = "prompt_length"
            record["target_prompt_len"] = prompt_len
            record["actual_input_tokens"] = actual_input_tokens
            record["fixed_output_len"] = fixed_output_len
            record["run_idx"] = run_idx

            records.append(record)

    return records


# =========================
# Sweep 2: TPOT vs output length
# =========================

def run_output_length_sweep(
    model,
    tokenizer,
    cfg: SweepConfig,
    fixed_prompt_len: int = 128,
) -> List[Dict[str, Any]]:
    """
    Sweep output lengths while fixing prompt length.
    Goal: TPOT vs output length
    """
    records = []

    base_prompt = build_base_prompt()
    prompt = make_prompt_near_target_tokens(
        tokenizer=tokenizer,
        base_text=base_prompt,
        target_prompt_len=fixed_prompt_len,
    )
    actual_input_tokens = count_input_tokens(tokenizer, prompt)

    print(f"\n=== Fixed prompt for output sweep: target={fixed_prompt_len}, actual={actual_input_tokens} ===")

    for output_len in cfg.output_lengths:
        print(f"\n=== Output sweep: max_new_tokens={output_len} ===")

        # optional: warmup per setting
        # TODO:
        # decide whether to warm up once globally or once per setting

        for run_idx in range(cfg.num_runs):
            print(f"[Output sweep run {run_idx+1}/{cfg.num_runs}]")

            record = run_prefill_and_decode_measurement(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                max_new_tokens=output_len,
                cfg=cfg,
            )

            record["sweep_type"] = "output_length"
            record["fixed_prompt_len"] = fixed_prompt_len
            record["actual_input_tokens"] = actual_input_tokens
            record["target_output_len"] = output_len
            record["run_idx"] = run_idx

            records.append(record)

    return records


# =========================
# Aggregation
# =========================

def group_mean(records: List[Dict[str, Any]], key: str, metric: str) -> List[Dict[str, Any]]:
    """
    Compute mean(metric) grouped by key.
    Example:
      group_mean(records, key="target_prompt_len", metric="ttft_sec")
    """
    # TODO:
    # 1. group records by key
    # 2. average metric per group
    # 3. return sorted list like:
    # [
    #   {"x": 32, "metric_mean": ...},
    #   {"x": 128, "metric_mean": ...},
    # ]
    raise NotImplementedError


def save_json(obj, out_path: str):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# =========================
# Plotting
# =========================

def plot_ttft_vs_prompt_length(
    prompt_records: List[Dict[str, Any]],
    out_path: str,
):
    """
    Plot TTFT vs prompt length
    """
    grouped = group_mean(
        prompt_records,
        key="target_prompt_len",
        metric="ttft_sec",
    )

    # TODO:
    # 1. extract x and y
    # 2. create matplotlib plot
    # 3. label axes
    # 4. save figure to out_path
    raise NotImplementedError


def plot_tpot_vs_output_length(
    output_records: List[Dict[str, Any]],
    out_path: str,
):
    """
    Plot TPOT vs output length
    """
    grouped = group_mean(
        output_records,
        key="target_output_len",
        metric="tpot_sec",
    )

    # TODO:
    # 1. extract x and y
    # 2. create matplotlib plot
    # 3. label axes
    # 4. save figure to out_path
    raise NotImplementedError


# =========================
# Optional analysis helpers
# =========================

def summarize_prompt_sweep(prompt_records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Create a compact summary for prompt sweep.
    """
    # TODO:
    # include at least:
    # - prompt length -> avg TTFT
    # - maybe total latency too
    raise NotImplementedError


def summarize_output_sweep(output_records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Create a compact summary for output sweep.
    """
    # TODO:
    # include at least:
    # - output length -> avg TPOT
    # - maybe total latency too
    raise NotImplementedError


# =========================
# Main
# =========================

def main():
    cfg = SweepConfig()

    print("=== Day 4 Sweep Config ===")
    print(cfg)

    ensure_dir(cfg.results_dir)

    tokenizer, model = load_tokenizer_and_model(cfg)

    # TODO:
    # Decide your warmup strategy.
    # Example:
    # - one global warmup with medium prompt/output
    # - or warmup inside each sweep
    #
    # Suggested medium warmup prompt:
    #   prompt len ~128
    # Suggested medium warmup output:
    #   max_new_tokens = 64

    prompt_records = run_prompt_length_sweep(
        model=model,
        tokenizer=tokenizer,
        cfg=cfg,
        fixed_output_len=64,
    )

    output_records = run_output_length_sweep(
        model=model,
        tokenizer=tokenizer,
        cfg=cfg,
        fixed_prompt_len=128,
    )

    prompt_summary = summarize_prompt_sweep(prompt_records)
    output_summary = summarize_output_sweep(output_records)

    prompt_records_path = str(Path(cfg.results_dir) / "prompt_sweep_records.json")
    output_records_path = str(Path(cfg.results_dir) / "output_sweep_records.json")
    prompt_summary_path = str(Path(cfg.results_dir) / "prompt_sweep_summary.json")
    output_summary_path = str(Path(cfg.results_dir) / "output_sweep_summary.json")

    save_json(prompt_records, prompt_records_path)
    save_json(output_records, output_records_path)
    save_json(prompt_summary, prompt_summary_path)
    save_json(output_summary, output_summary_path)

    plot_ttft_vs_prompt_length(
        prompt_records=prompt_records,
        out_path=str(Path(cfg.results_dir) / "ttft_vs_prompt_length.png"),
    )

    plot_tpot_vs_output_length(
        output_records=output_records,
        out_path=str(Path(cfg.results_dir) / "tpot_vs_output_length.png"),
    )

    print("\nSaved:")
    print(prompt_records_path)
    print(output_records_path)
    print(prompt_summary_path)
    print(output_summary_path)
    print(str(Path(cfg.results_dir) / "ttft_vs_prompt_length.png"))
    print(str(Path(cfg.results_dir) / "tpot_vs_output_length.png"))


if __name__ == "__main__":
    main()

Writing benchmark_minimal_skeleton.py


In [ ]:
!python benchmark_sweep_skeleton.py

=== Benchmark Config ===
BenchmarkConfig(model_name='Qwen/Qwen2.5-0.5B-Instruct', device='cuda', dtype='float16', max_new_tokens=64, num_runs=3, num_warmup=1, do_sample=False, temperature=1.0, top_p=1.0, use_kv_cache=True, results_dir='./results')
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:01<00:00, 199.47it/s, Materializing param=model.norm.weight]

=== Prompt Preview ===
You are a helpful AI assistant. Please explain the role of attention in transformer models, including why attention is useful for long-range dependency modeling. Add more explanation about self-attention, key-value interactions, and contextual representation learning. Add more explanation about self...

=== Warmup ===
[Warmup 1/1]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

=== Measured Runs ===
[Run 1/3]
{
  "run_idx": 0,
  "input_tokens": 128,
  "generated_tokens": 64